[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SekyungHan/ai-power-textbook-labs/blob/master/solutions/ch06_eda_tools.ipynb)

# Ch 6: AI 연구 도구 비교 — EDA 자동화 (정답)

In [ ]:
# Ch06: AI 연구 도구 비교 — EDA 자동화
import sys, subprocess
if 'google.colab' in sys.modules:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                          'pandapower'])
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                          'koreanize-matplotlib'])
import pandapower as pp
import pandapower.networks as pn
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
%matplotlib inline

np.random.seed(42)

try:
    sys.path.insert(0, '..')
    from utils.style import set_style, COLORS
    set_style()
except:
    # 한글 폰트 fallback (utils.style 없는 환경)
    import platform as _pf
    _sys = _pf.system()
    if _sys == 'Windows':
        plt.rcParams['font.family'] = 'Malgun Gothic'
    elif _sys == 'Darwin':
        plt.rcParams['font.family'] = 'AppleGothic'
    else:
        try:
            import koreanize_matplotlib
        except ImportError:
            plt.rcParams['font.family'] = 'NanumGothic'
    plt.rcParams['axes.unicode_minus'] = False


## 1. IEEE 14-bus EDA

In [ ]:
net = pn.case14()
pp.runpp(net)

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))

# (a) 모선별 전압 프로파일
buses = net.res_bus.index
voltages = net.res_bus.vm_pu.values
colors = ['#C75C3A' if v < 0.95 or v > 1.05 else '#5A7D6A' for v in voltages]
axes[0].bar(buses, voltages, color=colors, width=0.6)
axes[0].axhline(y=1.05, color='#C75C3A', ls='--', lw=0.8)
axes[0].axhline(y=0.95, color='#C75C3A', ls='--', lw=0.8)
axes[0].axhline(y=1.00, color='#2C3E50', ls='-', lw=0.5)
axes[0].set_xlabel('모선 번호')
axes[0].set_ylabel('전압 (p.u.)')
axes[0].set_title('(a) 모선 전압 프로파일')

# (b) 선로 부하율
loading = net.res_line.loading_percent.values
line_idx = net.res_line.index
axes[1].barh(line_idx, loading, color='#1B7A8A', height=0.6)
axes[1].axvline(x=100, color='#C75C3A', ls='--', lw=0.8)
axes[1].set_xlabel('부하율 (%)')
axes[1].set_ylabel('선로 번호')
axes[1].set_title('(b) 선로 부하율')

# (c) 발전기 출력
gen_p = net.res_gen.p_mw.values
axes[2].bar(range(len(gen_p)), gen_p, color='#D4984A', width=0.5)
axes[2].set_xlabel('발전기 번호')
axes[2].set_ylabel('유효전력 (MW)')
axes[2].set_title('(c) 발전기 출력')

plt.tight_layout()
plt.show()

## 2. Monte Carlo 시뮬레이션

In [ ]:
n_scenarios = 200
results = []

for i in range(n_scenarios):
    net_mc = pn.case14()
    scale = np.random.uniform(0.7, 1.3, len(net_mc.load))
    net_mc.load.p_mw *= scale
    net_mc.load.q_mvar *= scale
    
    try:
        pp.runpp(net_mc)
        results.append({
            'total_load': net_mc.res_load.p_mw.sum(),
            'min_voltage': net_mc.res_bus.vm_pu.min(),
            'max_loading': net_mc.res_line.loading_percent.max(),
            'total_loss': net_mc.res_line.pl_mw.sum(),
        })
    except:
        continue

df = pd.DataFrame(results)
print(f"수렴 시나리오: {len(df)}/{n_scenarios}")
print(df.describe())

## 3. 결과 시각화

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

axes[0,0].hist(df['min_voltage'], bins=30, color='#1B7A8A', edgecolor='white')
axes[0,0].axvline(x=0.95, color='#C75C3A', ls='--', label='하한 (0.95)')
axes[0,0].set_xlabel('최소 전압 (p.u.)')
axes[0,0].set_title('최소 전압 분포')
axes[0,0].legend()

axes[0,1].hist(df['max_loading'], bins=30, color='#D4984A', edgecolor='white')
axes[0,1].set_xlabel('최대 선로 부하율 (%)')
axes[0,1].set_title('최대 선로 부하율 분포')

axes[1,0].scatter(df['total_load'], df['min_voltage'], alpha=0.5, s=10, color='#2C3E50')
axes[1,0].set_xlabel('총 부하 (MW)')
axes[1,0].set_ylabel('최소 전압 (p.u.)')
axes[1,0].set_title('부하-전압 상관관계')

axes[1,1].scatter(df['total_load'], df['total_loss'], alpha=0.5, s=10, color='#5A7D6A')
axes[1,1].set_xlabel('총 부하 (MW)')
axes[1,1].set_ylabel('총 손실 (MW)')
axes[1,1].set_title('부하-손실 상관관계')

plt.tight_layout()
plt.show()